In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from shapely.geometry import (
    LineString,
    MultiLineString,
    MultiPoint,
    Point,
)
from shapely.ops import split, substring

In [ ]:
# Paths

case_dir = Path(
    r"C:\Users\NL1C2G\Sweco AB\51024754_Vallei_en_Veluwe_Afwateringseenheden - General\Analyse\3_analyses\leuvenumse_beek"
)

base_dir = case_dir / "0_basisdata"
result_dir = case_dir / "1_resultaten"
result_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Read input data

duikers = gpd.read_file(base_dir / "duikers.gpkg")
overige_watergangen = gpd.read_file(
    base_dir / "overige_watergang.gpkg"
)
hydroobjecten = gpd.read_file(base_dir / "hydroobject.gpkg")


In [ ]:
# Output paths

output_duikers = result_dir / "duikers_test.gpkg"
output_overige_watergangen = result_dir / "overige_watergangen_test.gpkg"
output_hydroobjecten = result_dir / "hydroobjecten_test.gpkg"

In [ ]:
# Settings

MAX_DISTANCE = 3.0  # metres
MIN_LINE_LENGTH = 1e-8

In [ ]:
# Check CRS

for name, gdf in {
    "duikers": duikers,
    "overige_watergangen": overige_watergangen,
    "hydroobjecten": hydroobjecten,
}.items():
    if gdf.crs is None:
        raise ValueError(f"{name} has no CRS.")

    if gdf.crs.is_geographic:
        raise ValueError(
            f"{name} uses a geographic CRS ({gdf.crs}). "
            "Reproject all data to a projected CRS with metres as units."
        )


# Use the CRS of the duikers as the working CRS
working_crs = duikers.crs

overige_watergangen = overige_watergangen.to_crs(working_crs)
hydroobjecten = hydroobjecten.to_crs(working_crs)


In [ ]:
# Keep only line geometries and explode multipart geometries

def prepare_lines(gdf):
    result = gdf.copy()

    result = result[
        result.geometry.notna()
        & ~result.geometry.is_empty
        & result.geometry.geom_type.isin(
            ["LineString", "MultiLineString"]
        )
    ].copy()

    result = result.explode(
        index_parts=False,
        ignore_index=True,
    )

    result = result[
        result.geometry.notna()
        & ~result.geometry.is_empty
        & (result.geometry.geom_type == "LineString")
        & (result.geometry.length > MIN_LINE_LENGTH)
    ].copy()

    return result.reset_index(drop=True)


other_lines = prepare_lines(overige_watergangen)
hydro_lines = prepare_lines(hydroobjecten)


In [ ]:
# Prepare duikers as individual LineStrings

duiker_lines = prepare_lines(duikers)

In [ ]:
# Create endpoint table for overige watergangen

endpoint_records = []

for line_id, row in other_lines.iterrows():
    line = row.geometry

    endpoint_records.append(
        {
            "line_id": line_id,
            "side": "start",
            "geometry": Point(line.coords[0]),
        }
    )

    endpoint_records.append(
        {
            "line_id": line_id,
            "side": "end",
            "geometry": Point(line.coords[-1]),
        }
    )

overige_endpoints = gpd.GeoDataFrame(
    endpoint_records,
    geometry="geometry",
    crs=working_crs,
)


# %%
# Create a combined list of all candidate network lines

network_lines = []

for line_id, row in other_lines.iterrows():
    network_lines.append(
        {
            "layer": "overige_watergangen",
            "line_id": line_id,
            "geometry": row.geometry,
        }
    )

for line_id, row in hydro_lines.iterrows():
    network_lines.append(
        {
            "layer": "hydroobjecten",
            "line_id": line_id,
            "geometry": row.geometry,
        }
    )


# %%
# Storage for split points and new connecting lines

split_points = {
    "overige_watergangen": {},
    "hydroobjecten": {},
}

# Store the new endpoint locations of the duikers
duiker_endpoint_targets = {}

connection_log = []

In [ ]:
# Helper functions

def nearest_endpoint(point, endpoint_gdf):
    """
    Return the nearest overige-watergang endpoint and its distance.
    """
    if endpoint_gdf.empty:
        return None, None

    distances = endpoint_gdf.geometry.distance(point)
    nearest_position = distances.idxmin()
    nearest_distance = distances.loc[nearest_position]

    return endpoint_gdf.loc[nearest_position], nearest_distance


def nearest_network_line(point, network_lines):
    """
    Find the nearest network line and project the input point
    exactly onto that line.
    """
    best_record = None
    best_projection = None
    best_distance = float("inf")

    for record in network_lines:
        line = record["geometry"]

        distance = line.distance(point)

        if distance < best_distance:
            # Distance along the line from its start
            distance_along_line = line.project(point)

            # Exact point on the original network line
            target_point = line.interpolate(
                distance_along_line
            )

            best_record = record
            best_projection = target_point
            best_distance = distance

    return (
        best_record,
        best_projection,
        best_distance,
    )


def add_split_point(layer_name, line_id, point):
    """
    Store a point where an existing line must be split.
    """
    split_points[layer_name].setdefault(line_id, [])

    # Avoid storing duplicate split points
    for existing_point in split_points[layer_name][line_id]:
        if existing_point.distance(point) < 1e-8:
            return

    split_points[layer_name][line_id].append(point)


In [ ]:
# %%
# Process every duiker endpoint and extend the duiker geometry

for duiker_id, row in duiker_lines.iterrows():
    duiker_line = row.geometry

    original_coordinates = list(duiker_line.coords)

    original_start = Point(original_coordinates[0])
    original_end = Point(original_coordinates[-1])

    # These remain None when no connection is found
    start_connection_point = None
    end_connection_point = None

    duiker_endpoints = [
        (0, original_start),
        (1, original_end),
    ]

    for endpoint_number, duiker_endpoint in duiker_endpoints:

        # --------------------------------------------------------------
        # 1. Find nearest endpoint of an overige watergang
        # --------------------------------------------------------------
        nearest_ep, endpoint_distance = nearest_endpoint(
            duiker_endpoint,
            overige_endpoints,
        )

        if (
            nearest_ep is not None
            and endpoint_distance <= MAX_DISTANCE
        ):
            target_point = nearest_ep.geometry
            target_layer = "overige_watergangen"
            target_line_id = nearest_ep.line_id
            match_type = "endpoint_overige_watergang"
            connection_distance = endpoint_distance

        else:
            # ----------------------------------------------------------
            # 2. Find nearest watergang or hydroobject line
            # ----------------------------------------------------------
            (
                nearest_line,
                target_point,
                line_distance,
            ) = nearest_network_line(
                duiker_endpoint,
                network_lines,
            )

            if (
                nearest_line is None
                or line_distance > MAX_DISTANCE
            ):
                connection_log.append(
                    {
                        "duiker_id": duiker_id,
                        "endpoint_number": endpoint_number,
                        "status": "geen verbinding gevonden",
                        "distance": (
                            line_distance
                            if nearest_line is not None
                            else None
                        ),
                    }
                )
                continue

            target_layer = nearest_line["layer"]
            target_line_id = nearest_line["line_id"]
            match_type = "projectie_op_lijn"
            connection_distance = line_distance

            # The network line is split at the projection point
            add_split_point(
                target_layer,
                target_line_id,
                target_point,
            )

        # --------------------------------------------------------------
        # 3. Store the connection point
        #
        # Do not replace the original endpoint.
        # The point will be added before or after the original duiker.
        # --------------------------------------------------------------
        if endpoint_number == 0:
            start_connection_point = target_point
        else:
            end_connection_point = target_point

        connection_log.append(
            {
                "duiker_id": duiker_id,
                "endpoint_number": endpoint_number,
                "status": "verbonden",
                "match_type": match_type,
                "target_layer": target_layer,
                "target_line_id": target_line_id,
                "distance": connection_distance,
            }
        )

    # --------------------------------------------------------------
    # 4. Extend the original duiker geometry
    # --------------------------------------------------------------
    new_coordinates = original_coordinates.copy()

    # Add the start connection before the original start point
    if (
        start_connection_point is not None
        and start_connection_point.distance(original_start)
        > MIN_LINE_LENGTH
    ):
        new_coordinates.insert(
            0,
            tuple(start_connection_point.coords)[0],
        )

    # Add the end connection after the original end point
    if (
        end_connection_point is not None
        and end_connection_point.distance(original_end)
        > MIN_LINE_LENGTH
    ):
        new_coordinates.append(
            tuple(end_connection_point.coords)[0]
        )

    extended_duiker = LineString(new_coordinates)

    duiker_lines.loc[duiker_id, "geometry"] = extended_duiker

In [ ]:
# %%
# Split original network lines exactly at projected points

def split_network_lines(gdf, layer_name):
    """
    Split lines exactly at the stored projection points.

    The split points are converted to distances along each line.
    The line is then reconstructed using shapely.ops.substring().
    """

    output_rows = []

    for line_id, row in gdf.iterrows():
        line = row.geometry

        if line is None or line.is_empty:
            continue

        points = split_points[layer_name].get(line_id, [])

        # Start and end distances of the line
        split_distances = [0.0, float(line.length)]

        for point in points:
            # Project the point again onto this exact line.
            # This guarantees that the split location lies exactly
            # on the line, avoiding floating-point errors.
            projected_point = line.interpolate(
                line.project(point)
            )

            distance_along_line = line.project(projected_point)

            # Ignore points at the existing endpoints
            if (
                distance_along_line > MIN_LINE_LENGTH
                and distance_along_line
                < line.length - MIN_LINE_LENGTH
            ):
                split_distances.append(
                    float(distance_along_line)
                )

        # Remove duplicate split distances and sort them
        split_distances = sorted(
            set(
                round(distance, 12)
                for distance in split_distances
            )
        )

        # Create each line part between two consecutive distances
        geometries = []

        for start_distance, end_distance in zip(
            split_distances[:-1],
            split_distances[1:],
        ):
            if (
                end_distance - start_distance
                <= MIN_LINE_LENGTH
            ):
                continue

            geometry = substring(
                line,
                start_distance,
                end_distance,
                normalized=False,
            )

            if (
                geometry is not None
                and not geometry.is_empty
                and geometry.length > MIN_LINE_LENGTH
            ):
                geometries.append(geometry)

        # If no split was necessary, retain the original line
        if not geometries:
            geometries = [line]

        for part_number, geometry in enumerate(geometries):
            new_row = row.copy()

            new_row["geometry"] = geometry
            new_row["is_connectie"] = False
            new_row["connectie_type"] = None
            new_row["bron_line_id"] = line_id
            new_row["deelnummer"] = part_number

            output_rows.append(new_row)

    if not output_rows:
        return gpd.GeoDataFrame(
            columns=list(gdf.columns)
            + [
                "is_connectie",
                "connectie_type",
                "bron_line_id",
                "deelnummer",
                "geometry",
            ],
            geometry="geometry",
            crs=gdf.crs,
        )

    return gpd.GeoDataFrame(
        output_rows,
        geometry="geometry",
        crs=gdf.crs,
    ).reset_index(drop=True)

In [ ]:
def make_codes_unique(gdf):
    """
    Add a suffix to duplicate code values.

    For example:
        ABC
        ABC

    becomes:
        ABC-0
        ABC-1
    """
    result = gdf.copy()

    if "code" not in result.columns:
        return result

    result["code"] = result["code"].astype(str)

    duplicate_codes = result["code"].duplicated(
        keep=False
    )

    occurrence = result.groupby(
        "code",
        sort=False,
    ).cumcount()

    result.loc[duplicate_codes, "code"] = (
        result.loc[duplicate_codes, "code"]
        + "-"
        + occurrence.loc[duplicate_codes].astype(str)
    )

    return result

split_other_lines = split_network_lines(
    other_lines,
    "overige_watergangen",
)

split_hydro_lines = split_network_lines(
    hydro_lines,
    "hydroobjecten",
)

In [ ]:

# Append new connecting lines to their target layer

def append_connections(
    split_lines,
    connection_records,
    layer_name,
):
    if not connection_records:
        return split_lines

    new_rows = []

    for connection_id, record in enumerate(connection_records):
        new_row = {
            column: None
            for column in split_lines.columns
            if column != "geometry"
        }

        new_row["geometry"] = record["geometry"]
        new_row["is_connectie"] = True
        new_row["connectie_type"] = record["connectie_type"]
        new_row["bron_line_id"] = record["bron_line_id"]
        new_row["deelnummer"] = connection_id

        new_rows.append(new_row)

    connections_gdf = gpd.GeoDataFrame(
        new_rows,
        geometry="geometry",
        crs=split_lines.crs,
    )

    return pd.concat(
        [split_lines, connections_gdf],
        ignore_index=True,
    )


output_other_lines = split_other_lines
output_hydro_lines = split_hydro_lines

output_other_lines = make_codes_unique(
    output_other_lines
)

output_hydro_lines = make_codes_unique(
    output_hydro_lines
)

In [ ]:
# Prepare output duikers

output_duiker_gdf = duiker_lines.copy()

if output_duiker_gdf.crs != working_crs:
    output_duiker_gdf = output_duiker_gdf.to_crs(working_crs)

In [ ]:
# Write output files

# Remove existing files if they already exist
for output_file in [
    output_duikers,
    output_overige_watergangen,
    output_hydroobjecten,
]:
    if output_file.exists():
        output_file.unlink()


output_duiker_gdf.to_file(
    output_duikers,
    layer="duikers",
    driver="GPKG",
)

output_other_lines.to_file(
    output_overige_watergangen,
    layer="overige_watergangen",
    driver="GPKG",
)

output_hydro_lines.to_file(
    output_hydroobjecten,
    layer="hydroobjecten",
    driver="GPKG",
)


In [ ]:
# Save a connection log for quality control

connection_log_gdf = pd.DataFrame(connection_log)

connection_log_path = result_dir / "duiker_verbindingen_log.csv"
connection_log_gdf.to_csv(
    connection_log_path,
    index=False,
)

print("Processing complete.")
print(f"Duikers: {output_duikers}")
print(f"Overige watergangen: {output_overige_watergangen}")
print(f"Hydroobjecten: {output_hydroobjecten}")
print(f"Connection log: {connection_log_path}")

print()
print(connection_log_gdf["status"].value_counts(dropna=False))